In [1]:
import json

In [4]:
# path to your OCEL file (e.g. .json, .jsonocel, .xml, .xmlocel)
ocel_path = "../Logs/angular_github_commits_ocel.jsonocel"

ocel = {"events": [], "objects": []}

if ocel_path.endswith(".json") or ocel_path.endswith(".jsonocel") or ocel_path.endswith(".jsonocel"):
    with open(ocel_path, "r", encoding="utf-8") as f:
        ocel_data = json.load(f)
        ocel["events"] = ocel_data.get("ocel:events", {})
        if isinstance(ocel["events"], dict):
            # some versions use dict for events, some use list
            ocel["events"] = [dict(v, id=k) for k, v in ocel["events"].items()]
        elif not ocel["events"]:
            ocel["events"] = ocel_data.get("events", [])
            
        ocel["objects"] = ocel_data.get("ocel:objects", {})
        if isinstance(ocel["objects"], dict):
            ocel["objects"] = [dict(v, id=k) for k, v in ocel["objects"].items()]
        elif not ocel["objects"]:
            ocel["objects"] = ocel_data.get("objects", [])

elif ocel_path.endswith(".xml") or ocel_path.endswith(".xmlocel"):
    import xml.etree.ElementTree as ET
    tree = ET.parse(ocel_path)
    root = tree.getroot()
    
    # Parse events
    events_node = root.find(".//events")
    if events_node is not None:
        for ev in events_node.findall("event"):
            # Attributes like 'id', 'type', 'activity', 'time'
            ev_dict = dict(ev.attrib)
            # You can also parse child nodes here if needed
            ocel["events"].append(ev_dict)
            
    # Parse objects
    objects_node = root.find(".//objects")
    if objects_node is not None:
        # In OCEL 2.0 XML objects are often top-level under <objects>
        for obj in objects_node.findall("object"):
            # Attributes like 'id', 'type'
            obj_dict = dict(obj.attrib)
            ocel["objects"].append(obj_dict)


In [5]:
# Get all event names (checking for 'type', 'activity', or 'ocel:activity' attributes)
# This collects unique values and preserves string representation.
event_names = sorted({
    e.get("type") or e.get("activity") or e.get("ocel:activity") 
    for e in ocel.get("events", []) 
    if (e.get("type") or e.get("activity") or e.get("ocel:activity")) is not None
})
print("Event names:", event_names)

Event names: ['add', 'address', 'aio', 'benchmark', 'bug', 'build', 'bump', 'check', 'chore', 'ci', 'cleanup', 'code', 'condition', 'configure', 'consolidated', 'copy', 'design', 'developer.md', 'docs', "don't", 'edit', 'example', 'examples', 'feat', 'filetree', 'finished', 'first', 'fix', 'form', 'format', 'include', 'initial', 'keep', 'make', 'merge', 'misc', 'more', 'moved', 'patch', 'perf', 'produce', 'read', 'readme', 'refactor', 'release', 'remove', 'rendererv2', 'repackaging', 'replaced', 'resolve', 'revert', 'roll', 'router', 'squash', 'squash-me', 'style', 'table', 'temporary', 'test', 'text', 'tools', 'transpiler', 'tweak', 'typo', 'update', 'use', 'wip']


In [6]:
# Get object names: OCEL distinguishes object types and object IDs.
# If objects have a 'name' attribute, collect that; otherwise show type:ID.
object_names = set()
unique_objects = set()  # to track unique object types
for obj in ocel.get("objects", []):
    obj_id = obj.get("id", "UNKNOWN_ID")
    # object type stored in 'type' or 'ocel:type'
    otype = obj.get("type") or obj.get("ocel:type") or "UNKNOWN_TYPE"
    
    # optional human-readable name attribute
    oname = obj.get("name") or obj.get("ocel:name") or f"{otype}:{obj_id}"
    obj_type, obj_id = oname.split(":") if ":" in oname else (oname, "UNKNOWN_ID")
    unique_objects.add(obj_type)
    object_names.add((obj_type, obj_id))

object_names = sorted(object_names)
print("Object types:", unique_objects)

Object types: {'files', 'branches'}


In [ ]:
unique_objects

{'CreditNote',
 'Customer',
 'Dispute',
 'Invoice',
 'Order',
 'Payment',
 'Product',
 'Return',
 'Shipment'}